# Q-IQL: Barren Plateau Empirical Study on Helios

## Design

For each shot, the circuit uses a unique RNG seed (`get_current_shot()`) to generate random angles for all parameters. Each shot is therefore a completely independent random initialisation.

**Metric:** Var[<Z0>] over N shots with random parameters.
If it decays exponentially with n -> barren plateau present.
If it stays stable/polynomial -> BP-safe design confirmed.

**Configs** (all BP-safe: L <= floor(log2(n))):
- n=4 L=1 / n=4 L=2 / n=8 L=1 / n=8 L=3 (full Q-IQL)

**Backends:** Selene (noiseless) | Helios-1E (QSystemErrorModel) | Helios HW (n=8 L=3 only)

In [2]:
import datetime
import json
import numpy as np
import qnexus as qnx
from guppylang import guppy
from guppylang.std.qsystem import phased_x, rz, zz_phase, measure, qubit
from guppylang.std.qsystem.utils import get_current_shot
from guppylang.std.qsystem.random import RNG
from guppylang.std.builtins import result, comptime
from guppylang.std.angles import angle
from guppylang.std.quantum import discard

print("Libraries loaded OK")

Libraries loaded OK


In [3]:
project = qnx.projects.get_or_create(name='quantum-iql-barren-plateau')
qnx.context.set_active_project(project)
suffix = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
print(f'Project ready. Suffix: {suffix}')

Project ready. Suffix: 20260520-060821


In [ ]:
CONFIGS           = [(4, 1), (4, 2), (8, 1), (8, 3)]
HELIOS_CONFIG     = (8, 3)
N_SHOTS_PER_INIT  = 50
N_INIT            = 150
N_SHOTS           = N_SHOTS_PER_INIT * N_INIT
INIT_SEED         = 42

def hqc_estimate(n_qubits, n_layers, n_shots):
    n_1q = n_layers * n_qubits * 2
    n_2q = (n_layers + 1) * (n_qubits - 1)
    n_m  = 1
    return 5 + (n_1q + 10 * n_2q + 5 * n_m) * n_shots / 5000

print(f'HQC cost estimates ({N_SHOTS} shots per job = {N_INIT} inits x {N_SHOTS_PER_INIT} shots each):')
total_emu = 0.0
for n, L in CONFIGS:
    cost = hqc_estimate(n, L, N_SHOTS)
    total_emu += cost * 2
    print(f'  n={n} L={L}: ~{cost:.1f} HQC')
cost_hw = hqc_estimate(*HELIOS_CONFIG, N_SHOTS)
print(f'\nTotal emulator (Selene + Helios-1E): ~{total_emu:.1f} HQC (budget: 10,000)')
print(f'Total Helios HW (n=8 L=3 only)    : ~{cost_hw:.1f} HQC (budget: 530)')
ok = total_emu <= 10000 and cost_hw <= 530
print('\n' + ('OK Within budget' if ok else 'OVER BUDGET'))

HQC cost estimates (7500 shots per job = 150 inits x 50 shots each):
  n=4 L=1: ~114.5 HQC
  n=4 L=2: ~171.5 HQC
  n=8 L=1: ~246.5 HQC
  n=8 L=3: ~504.5 HQC

Total emulator (Selene + Helios-1E): ~2074.0 HQC (budget: 10,000)
Total Helios HW (n=8 L=3 only)    : ~504.5 HQC (budget: 530)

OK Within budget


In [5]:
import tempfile, importlib.util, os

def build_rng_dru(n_qubits, n_layers, init_seed, n_shots_per_init):
    """
    DRU circuit where shots are grouped in blocks of n_shots_per_init.
    All shots in block i share the same RNG seed -> same random angles.
    seed = init_seed + shot_index // n_shots_per_init
    This allows estimating <Z0> per initialisation from multiple shots.
    Angles are in half-turns: angle(x) = x * pi radians.
    """
    qv    = [f'q{i}' for i in range(n_qubits)]
    alloc = ', '.join(qv)
    rhs   = ', '.join(['qubit()'] * n_qubits)

    lines = [
        'from guppylang import guppy',
        'from guppylang.std.qsystem import phased_x, rz, zz_phase, measure, qubit',
        'from guppylang.std.qsystem.utils import get_current_shot',
        'from guppylang.std.qsystem.random import RNG',
        'from guppylang.std.builtins import result, comptime',
        'from guppylang.std.angles import angle',
        'from guppylang.std.quantum import discard',
        '',
        '@guppy',
        'def main() -> None:',
        f'    rng = RNG(comptime({init_seed}) + get_current_shot() // comptime({n_shots_per_init}))',
        f'    {alloc} = {rhs}',
        '',
        '    # CZ preamble with random angles',
    ]
    for q in range(0, n_qubits - 1, 2):
        lines.append(f'    zz_phase(q{q}, q{q+1}, angle(rng.random_float() * 2.0))')
    for q in range(1, n_qubits - 1, 2):
        lines.append(f'    zz_phase(q{q}, q{q+1}, angle(rng.random_float() * 2.0))')
    for layer in range(n_layers):
        lines.append('')
        lines.append(f'    # DRU layer {layer + 1}')
        for q in range(n_qubits):
            lines.append(f'    rz(q{q}, angle(rng.random_float() * 2.0))')
            lines.append(f'    phased_x(q{q}, angle(rng.random_float() * 2.0), angle(rng.random_float() * 2.0))')
            lines.append(f'    rz(q{q}, angle(rng.random_float() * 2.0))')
        for q in range(n_qubits - 1):
            lines.append(f'    zz_phase(q{q}, q{q+1}, angle(rng.random_float() * 2.0))')
    lines += ['', '    bit0 = measure(q0)', "    result('V', bit0)"]
    for q in range(1, n_qubits):
        lines.append(f'    discard(q{q})')
    lines += ['    rng.discard()', '', 'hugr_pkg = main.compile()']

    src = '\n'.join(lines)
    with tempfile.NamedTemporaryFile(suffix='.py', delete=False, mode='w') as f:
        f.write(src)
        tmpfile = f.name
    try:
        spec = importlib.util.spec_from_file_location('circuit', tmpfile)
        mod  = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return mod.hugr_pkg
    finally:
        os.unlink(tmpfile)

print('Building RNG-based DRU circuits...')
hugr_packages = {}
for n, L in CONFIGS:
    hugr_packages[(n, L)] = build_rng_dru(n, L, INIT_SEED, N_SHOTS_PER_INIT)
    print(f'  n={n} L={L} -> compiled OK')
print('Done.')

Building RNG-based DRU circuits...
  n=4 L=1 -> compiled OK
  n=4 L=2 -> compiled OK
  n=8 L=1 -> compiled OK
  n=8 L=3 -> compiled OK
Done.


In [6]:
hugr_refs = {}
for (n, L), pkg in hugr_packages.items():
    ref = qnx.hugr.upload(hugr_package=pkg, name=f'dru-rng-n{n}-L{L}-{suffix}')
    hugr_refs[(n, L)] = ref
    print(f'  Uploaded n={n} L={L}')

  Uploaded n=4 L=1
  Uploaded n=4 L=2
  Uploaded n=8 L=1
  Uploaded n=8 L=3


In [7]:
def extract_z0(res):
    """Extract all binary outcomes as +1/-1 values."""
    return [1 - 2 * int(val)
            for shot in res.results
            for name, val in shot.entries
            if name == 'V']

def group_into_inits(z0_raw, n_shots_per_init, n_init):
    """Group raw binary shots into blocks -> one <Z0> estimate per init."""
    return [np.mean(z0_raw[i * n_shots_per_init : (i+1) * n_shots_per_init])
            for i in range(n_init)]

def report(label, z0_vals):
    n    = len(z0_vals)
    mean = np.mean(z0_vals)
    var  = np.var(z0_vals)
    ci95 = 1.96 * np.std(z0_vals) / np.sqrt(n)
    print(f'  {label}: mean={mean:.3f}  var={var:.4f}  CI95=+-{ci95:.4f}  (N={n} inits)')
    return {'mean': mean, 'var': var, 'ci95': ci95, 'n': n}

print('Helper functions ready.')

Helper functions ready.


In [32]:
selene_job_refs = {}
for (n, L), ref in hugr_refs.items():
    config = qnx.models.SeleneConfig(
        n_qubits=n, simulator=qnx.models.StatevectorSimulator())
    job_ref = qnx.start_execute_job(
        programs=[ref], n_shots=[N_SHOTS], backend_config=config,
        name=f'bp-selene-n{n}-L{L}-{suffix}')
    selene_job_refs[(n, L)] = job_ref
    print(f'  Submitted Selene n={n} L={L} ({N_SHOTS} shots)')
print('\nAll Selene jobs submitted.')

  Submitted Selene n=4 L=1 (7500 shots)
  Submitted Selene n=4 L=2 (7500 shots)
  Submitted Selene n=8 L=1 (7500 shots)
  Submitted Selene n=8 L=3 (7500 shots)

All Selene jobs submitted.


In [33]:
selene_results = {}
print('Selene (noiseless):')
for (n, L), job_ref in selene_job_refs.items():
    qnx.jobs.wait_for(job_ref, timeout=None)
    res = qnx.jobs.results(job_ref)[0].download_result()
    z0_raw = extract_z0(res)
    z0     = group_into_inits(z0_raw, N_SHOTS_PER_INIT, N_INIT)
    selene_results[(n, L)] = z0
    report(f'n={n} L={L}', z0)
print('\nSelene done.')

Selene (noiseless):
  n=4 L=1: mean=-0.114  var=0.4919  CI95=+-0.1122  (N=150 inits)
  n=4 L=2: mean=-0.055  var=0.3680  CI95=+-0.0971  (N=150 inits)
  n=8 L=1: mean=-0.059  var=0.4951  CI95=+-0.1126  (N=150 inits)
  n=8 L=3: mean=0.008  var=0.2841  CI95=+-0.0853  (N=150 inits)

Selene done.


In [36]:
helios1e_job_refs = {}
for (n, L), ref in hugr_refs.items():
    config = qnx.models.HeliosConfig(
        system_name='Helios-1E',
        max_cost=cost * 1.2,
        emulator_config=qnx.models.HeliosEmulatorConfig(
            n_qubits=n, error_model=qnx.models.QSystemErrorModel()))
    job_ref = qnx.start_execute_job(
        programs=[ref], n_shots=[N_SHOTS], backend_config=config,
        name=f'bp-helios1e-n{n}-L{L}-{suffix}')
    helios1e_job_refs[(n, L)] = job_ref
    print(f'  Submitted Helios-1E n={n} L={L} ({N_SHOTS} shots)')
print('\nAll Helios-1E jobs submitted.')

  Submitted Helios-1E n=4 L=1 (7500 shots)
  Submitted Helios-1E n=4 L=2 (7500 shots)
  Submitted Helios-1E n=8 L=1 (7500 shots)
  Submitted Helios-1E n=8 L=3 (7500 shots)

All Helios-1E jobs submitted.


In [37]:
helios1e_results = {}
print('Helios-1E (noisy emulator):')
for (n, L), job_ref in helios1e_job_refs.items():
    qnx.jobs.wait_for(job_ref, timeout=None)
    res = qnx.jobs.results(job_ref)[0].download_result()
    z0_raw = extract_z0(res)
    z0     = group_into_inits(z0_raw, N_SHOTS_PER_INIT, N_INIT)
    helios1e_results[(n, L)] = z0
    report(f'n={n} L={L}', z0)
print('\nHelios-1E done.')

Helios-1E (noisy emulator):


Unknown OpType in BackendInfo: `RZZ`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `Rxxyyzz`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `U1q`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `ZZ`, will omit from BackendInfo. Consider updating your pytket version.


  n=4 L=1: mean=-0.123  var=0.4996  CI95=+-0.1131  (N=150 inits)


Unknown OpType in BackendInfo: `RZZ`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `Rxxyyzz`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `U1q`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `ZZ`, will omit from BackendInfo. Consider updating your pytket version.


  n=4 L=2: mean=-0.065  var=0.3617  CI95=+-0.0962  (N=150 inits)


Unknown OpType in BackendInfo: `RZZ`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `Rxxyyzz`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `U1q`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `ZZ`, will omit from BackendInfo. Consider updating your pytket version.


  n=8 L=1: mean=-0.057  var=0.4803  CI95=+-0.1109  (N=150 inits)


Unknown OpType in BackendInfo: `RZZ`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `Rxxyyzz`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `U1q`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `ZZ`, will omit from BackendInfo. Consider updating your pytket version.


  n=8 L=3: mean=0.001  var=0.2820  CI95=+-0.0850  (N=150 inits)

Helios-1E done.


In [ ]:
RUN_ON_HARDWARE = True
helios_results  = {}
if RUN_ON_HARDWARE:
    n, L = HELIOS_CONFIG
    ref  = hugr_refs[HELIOS_CONFIG]
    cost = hqc_estimate(n, L, N_SHOTS)
    config = qnx.models.HeliosConfig(system_name='Helios-1', max_cost=cost)
    job_ref = qnx.start_execute_job(
        programs=[ref], n_shots=[N_SHOTS], backend_config=config,
        name=f'bp-helios-hw-n{n}-L{L}-{suffix}')
    print(f'Submitted Helios HW n={n} L={L} ({N_SHOTS} shots, ~{cost:.1f} HQC)')
    qnx.jobs.wait_for(job_ref, timeout=None)
    res = qnx.jobs.results(job_ref)[0].download_result()
    z0_raw = extract_z0(res)
    z0     = group_into_inits(z0_raw, N_SHOTS_PER_INIT, N_INIT)
    helios_results[HELIOS_CONFIG] = z0
    print('Helios HW:')
    report(f'n={n} L={L}', z0)
else:
    print('Hardware run skipped.')

Submitted Helios HW n=8 L=3 (7500 shots, ~504.5 HQC)


In [ ]:
import json
import numpy as np

data = json.load(open('Helios-1 Execution Result.json'))

z0_raw = [1 - 2 * int(item[0][1]) 
          for item in data 
          if len(item) > 0 and len(item[0]) >= 2 and item[0][0] == 'V']
print(f"Shots completados: {len(z0_raw)}")

n_inits_completed = len(z0_raw) // N_SHOTS_PER_INIT
print(f"Inits completas: {n_inits_completed}")

z0 = group_into_inits(z0_raw, N_SHOTS_PER_INIT, n_inits_completed)
report(f"n=8 L=3 HW (partial, {n_inits_completed} inits)", z0)
helios_results[HELIOS_CONFIG] = z0

Shots completados: 7156
Inits completas: 143
  n=8 L=3 HW (partial, 143 inits): mean=-0.021  var=0.2727  CI95=+-0.0856  (N=143 inits)


In [ ]:
summary = []
print(f"{'Config':<12} {'Selene Var':>12} {'H-1E Var':>12} {'HW Var':>10} {'Noise ratio':>12}")
print('-' * 62)
for (n, L) in sorted(selene_results.keys()):
    s_var   = np.var(selene_results[(n, L)])
    h1e_var = np.var(helios1e_results.get((n, L), [np.nan]))
    hw_var  = np.var(helios_results.get((n, L), [np.nan]))
    ratio   = h1e_var / s_var if s_var > 0 else float('nan')
    hw_s    = f'{hw_var:.5f}' if not np.isnan(hw_var) else '--'
    print(f'n={n} L={L}  {s_var:>12.5f} {h1e_var:>12.5f} {hw_s:>10} {ratio:>12.3f}')
    summary.append({'n': n, 'L': L, 'selene_var': s_var,
                    'helios1e_var': h1e_var, 'helios_hw_var': hw_var, 'noise_ratio': ratio})
with open('barren_plateau_results.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\nSaved -> barren_plateau_results.json')
print('\nInterpretation:')
print('  Var -> 0 with n: barren plateau present')
print('  Var stable/polynomial with n: BP-safe design confirmed')
print('  noise_ratio > 1: noise amplifies variance (noise-induced BP risk)')
print('  noise_ratio < 1: noise suppresses variance (noise-induced BP)')